# E1a — p8s-v2 hidden state 추출

깃의 팀 공용 프롬프트(`p8s_prompt.py`, p8s-v2)를 wget으로 받아 md5를 검증한 뒤 그대로 사용한다.
프롬프트를 이 노트북에서 재정의하지 않는다.

## 배경

E1a는 현경 담당이고 그 산출물로 E2a 프로브를 돌리는 것이 원래 순서다.
다만 9/20 기준 산출물을 아직 받지 못해, E2a 일정을 맞추기 위해 **hidden 추출만 우선 수행**한다.

- 이 노트북은 **hidden만 추출한다. K± 라벨 재판정은 하지 않는다.**
- `kside`는 CSV 값(v1 closed-book 기준)을 그대로 복사하므로, 이 산출물로 돌린 프로브 결과는
  **v2 hidden + v1 라벨 혼합 조건**이다. 보고 시 명시할 것.
- 라벨이 갱신되면 hidden은 그대로 두고 `kside`만 교체해 E2a를 재실행하면 된다. 재추출은 불필요하다.
- C± 라벨은 데이터 구성 시 정해진 값이라 프롬프트와 무관하다. 재판정 대상이 아니다.

## 규약

| 항목 | 값 |
|---|---|
| 추출 지점 | 프롬프트 마지막 토큰 (chat template의 generation prompt 끝) |
| 층 | `[N, 33, 4096]` — index 0은 embedding, 1~32가 decoder layer |
| truncation | 초과 시 **context만** 절단. generation prompt는 항상 보존 |
| padding | left padding → 마지막 토큰은 항상 `-1` |

런타임: **GPU (L4 이상 권장)**. 4bit 양자화 금지 — hidden 값이 기존 실험과 달라진다.

## 0. 환경

In [1]:
!pip -q install -U transformers accelerate

import os, gc, json, hashlib, subprocess, sys
import numpy as np, pandas as pd, torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임으로 변경할 것 (L4 이상 권장)")
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | {p.total_memory/1e9:.1f} GB")
if p.total_memory < 20e9:
    print("[경고] 8B bfloat16은 약 16GB 필요. T4(16GB)는 OOM 위험")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 145.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 40.5 MB/s eta 0:00:00
GPU: NVIDIA A100-SXM4-40GB | 42.4 GB


In [2]:
from google.colab import drive, userdata
drive.mount('/content/drive')

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    from huggingface_hub import login; login(token=os.environ["HF_TOKEN"])
    print("HF 인증 완료")
except Exception as e:
    print("보안 비밀에 HF_TOKEN이 없다:", e)
    from huggingface_hub import login; login()

Mounted at /content/drive
보안 비밀에 HF_TOKEN이 없다: Secret HF_TOKEN does not exist.


## 1. CONFIG

In [4]:
P9 = "/content/drive/MyDrive/Grad_share/data_share/P9/"

CONFIG = {
    "CSV":      P9 + "p8s_behavior.csv",   # 파일명 확인 필요 (p8s_pairs.csv일 수도)
    "OUT_DIR":  P9 + "E1a_v2_dy",          # 팀 경로와 겹치지 않게
    "MODES":    ["cq"],                    # ["cq", "q"] 가능
    "MAX_LEN":  1024,
    "BATCH":    4,                         # OOM 시 자동으로 절반씩 낮춤
    "PROMPT_URL": ("https://raw.githubusercontent.com/Ontology0/Graduation-Project/"
                   "refs/heads/dev/prompt/p8s_prompt.py"),
    "PROMPT_MD5": "20959a4c34203ea0866d030baf23595d",   # p8s-v2
}

COLMAP = {"question": "question", "context": "ctx_text",
          "k_label": "kside", "c_label": "ctx_label"}

os.makedirs(CONFIG["OUT_DIR"], exist_ok=True)
print("CSV 존재:", os.path.exists(CONFIG["CSV"]))

CSV 존재: False


In [5]:
import glob, os
P9 = "/content/drive/MyDrive/Grad_share/data_share/P9/"
print(os.path.exists(P9))
print(sorted(os.listdir(P9))[:40])

True
['E1a_v2_dy']


In [6]:
CONFIG["CSV"] = "/content/p8s_pairs.csv"
print(os.path.exists(CONFIG["CSV"]))

True


In [7]:
CONFIG["OUT_DIR"] = "/content/drive/MyDrive/p8s/E1a_v2_dy"
os.makedirs(CONFIG["OUT_DIR"], exist_ok=True)

## 2. 프롬프트 로드 + md5 검증

md5가 다르면 여기서 중단한다. 버전이 다른 결과끼리 비교하면 안 된다.

In [8]:
subprocess.run(["wget", "-q", "-O", "p8s_prompt.py", CONFIG["PROMPT_URL"]], check=True)
txt = open("p8s_prompt.py", "rb").read()
assert b"<title>" not in txt and b"PROMPT_VERSION" in txt, "HTML을 받았음 — URL 확인"

PROMPT_MD5 = hashlib.md5(txt).hexdigest()
assert PROMPT_MD5 == CONFIG["PROMPT_MD5"], (
    f"md5 불일치\n  기대 {CONFIG['PROMPT_MD5']}\n  실제 {PROMPT_MD5}\n"
    "팀에 확인 후 진행할 것")

sys.path.insert(0, "/content")
import p8s_prompt as P
print(f"{P.PROMPT_VERSION}  md5={PROMPT_MD5}  fewshot={len(P.FEWSHOT)}  "
      f"MAX_NEW_TOKENS={P.MAX_NEW_TOKENS}")
print(P.SYSTEM_PROMPT)

p8s-v2  md5=20959a4c34203ea0866d030baf23595d  fewshot=0  MAX_NEW_TOKENS=192
You are a question answering system. Answer the question based on the given context. Output the answer string itself and nothing else: no sentence, no explanation, no punctuation, no quotes. Keep it as short as possible (a name or a term).


## 3. 데이터 로드

In [9]:
df = pd.read_csv(CONFIG["CSV"])
print("shape:", df.shape)
print("columns:", list(df.columns))
missing = set(COLMAP.values()) - set(df.columns)
assert not missing, f"누락 컬럼: {missing}"

df["_qid"] = df[COLMAP["question"]].astype(str)
print("\nunique questions:", df._qid.nunique())
print(pd.crosstab(df[COLMAP["k_label"]], df[COLMAP["c_label"]]).to_string())

shape: (1224, 16)
columns: ['question', 'gold', 'prop', 'kside', 'knows', 'A_cb', 'ctx_id', 'ctx_title', 's_wiki_title', 'orig_gold', 'ctx_label', 'ctx_text', 'answer_surface', 'ans_start', 'ans_end', 'wrong_answer']

unique questions: 612
ctx_label   C+   C-
kside              
K+         378  378
K−         234  234


## 4. 프롬프트 빌더 (v2)

추출 지점이 마지막 토큰이므로 generation prompt는 어떤 경우에도 보존한다.

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(P.MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def build_ids(question, context=None, max_len=None):
    """추출용 token id. context=None이면 closed-book. 반환: (ids, 절단 문자 수)"""
    max_len = max_len or CONFIG["MAX_LEN"]
    if context is None:
        text = tok.apply_chat_template(P.build_messages_cb(question),
                                       tokenize=False, add_generation_prompt=True)
        ids = tok.encode(text, add_special_tokens=False)
        assert len(ids) <= max_len, f"closed-book이 max_len 초과: {question[:40]}"
        return ids, 0

    ctx, cut = str(context), 0
    while True:
        text = tok.apply_chat_template(P.build_messages(question, ctx),
                                       tokenize=False, add_generation_prompt=True)
        ids = tok.encode(text, add_special_tokens=False)
        if len(ids) <= max_len:
            return ids, cut
        drop = max(1, (len(ids) - max_len) * 4)   # 토큰 ≈ 4자 가정, 뒤에서부터
        assert drop < len(ctx), f"max_len이 context 없이도 부족: {question[:40]}"
        ctx, cut = ctx[:-drop], cut + drop

# 육안 확인 — v1(few-shot·ONLY)과 다른지 여기서 바로 보인다
ids, _ = build_ids(df._qid.iloc[0], df[COLMAP["context"]].iloc[0])
print(tok.decode(ids)[-500:])

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


a Movies banner and directed by K. Raghavendra Rao. It stars Jeetendra, Hema Malini, Rati Agnihotri in the pivotal roles and music composed by Laxmikant-Pyarelal. The film is a remake of the Telugu movie ""Kondaveeti Simham"" (1981), starring N. T. Rama Rao, Sridevi in the lead roles and both the movies are made under the same banner and director. ""Kondaveeti Simham"" itself was a remake of the 1974
Question: What genre is Kanoon?
Answer:<|eot_id|><|start_header_id|>assistant<|end_header_id|>




## 5. 토큰 길이 검증

C−는 문맥 속 정답 단어를 치환한 구조라, 절단 위치에 따라 치환어가 사라질 수 있다.

In [12]:
len_rows, trunc = [], []
for mode in CONFIG["MODES"]:
    src = df if mode == "cq" else df.drop_duplicates("_qid")
    qids = src["_qid"].tolist()
    ctxs = src[COLMAP["context"]].tolist() if mode == "cq" else [None] * len(src)
    for q, c in zip(qids, ctxs):
        ids, cut = build_ids(q, c)
        len_rows.append({"mode": mode, "qid": q, "n_tok": len(ids), "cut_chars": cut})
        if cut:
            trunc.append((mode, q, cut))

L = pd.DataFrame(len_rows)
print(L.groupby("mode")["n_tok"].agg(["min", "median", "max", "count"]).to_string())
L.to_csv(os.path.join(CONFIG["OUT_DIR"], "token_lengths.csv"), index=False)

if trunc:
    print(f"\n[경고] context 절단 {len(trunc)}행 — 치환어 소실 가능. 해당 qid 확인 필요")
    print([q for _, q, _ in trunc][:15])
else:
    print("\n[OK] truncation 없음")

      min  median  max  count
mode                         
cq    206   238.5  377   1224

[OK] truncation 없음


## 6. 모델 로드

In [13]:
try:
    model = AutoModelForCausalLM.from_pretrained(P.MODEL_ID, dtype=torch.bfloat16,
                                                 device_map="auto")
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(P.MODEL_ID, torch_dtype=torch.bfloat16,
                                                 device_map="auto")
model.eval()
model.config.use_cache = False        # prefill-only 추출
N_LAYERS = model.config.num_hidden_layers
D_MODEL  = model.config.hidden_size
print(f"layers={N_LAYERS}  dim={D_MODEL}")

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

layers=32  dim=4096


## 7. hidden 추출

left padding이므로 마지막 토큰은 항상 `-1`이다. OOM이 나면 batch를 절반씩 낮춘다.

In [14]:
@torch.no_grad()
def forward_batch(id_list):
    maxlen = max(len(x) for x in id_list)
    ii = torch.full((len(id_list), maxlen), tok.pad_token_id, dtype=torch.long)
    am = torch.zeros((len(id_list), maxlen), dtype=torch.long)
    for j, ids in enumerate(id_list):        # left padding
        ii[j, maxlen - len(ids):] = torch.tensor(ids)
        am[j, maxlen - len(ids):] = 1
    res = model(input_ids=ii.to(model.device), attention_mask=am.to(model.device),
                output_hidden_states=True, use_cache=False)
    out = torch.stack([h[:, -1, :] for h in res.hidden_states], dim=1).half().cpu()
    del res
    return out

@torch.no_grad()
def extract(id_lists, tag=""):
    out = torch.empty(len(id_lists), N_LAYERS + 1, D_MODEL, dtype=torch.float16)
    i, bs, step = 0, CONFIG["BATCH"], 0
    while i < len(id_lists):
        chunk = id_lists[i:i + bs]
        try:
            out[i:i + len(chunk)] = forward_batch(chunk)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); gc.collect()
            if bs == 1:
                raise
            bs = max(1, bs // 2); print(f"  [OOM] batch → {bs}", flush=True); continue
        i += len(chunk); step += 1
        if step % 25 == 0:
            print(f"  [{tag}] {i}/{len(id_lists)} (bs={bs})", flush=True)
            torch.cuda.empty_cache()
    return out

In [15]:
for mode in CONFIG["MODES"]:
    path = os.path.join(CONFIG["OUT_DIR"], f"hidden_{mode}.npz")
    if os.path.exists(path):
        print("[skip] 캐시 존재:", path); continue

    if mode == "cq":
        src = df
        ids_list = [build_ids(q, c)[0]
                    for q, c in zip(src["_qid"], src[COLMAP["context"]])]

    else:
        src = df.drop_duplicates("_qid")
        ids_list = [build_ids(q, None)[0] for q in src._qid]

    H = extract(ids_list, tag=mode)

    if mode == "q":                       # unique 질문 → 전체 행에 매핑
        idx = {q: i for i, q in enumerate(src._qid)}
        H = H[[idx[q] for q in df._qid]]

    np.savez_compressed(
        path,
        H=H.numpy(),
        question=df._qid.values.astype(str),
        kside=df[COLMAP["k_label"]].values.astype(str),
        ctx_label=df[COLMAP["c_label"]].values.astype(str),
        row_uid=np.arange(len(df)),
        prompt_version=np.array(P.PROMPT_VERSION),
        prompt_md5=np.array(PROMPT_MD5),
        mode=np.array(mode),
        layer_note=np.array("index 0 = embedding, 1..L = decoder layer"),
    )
    print(f"[saved] {path}  H={tuple(H.shape)}")
    del H; gc.collect(); torch.cuda.empty_cache()

with open(os.path.join(CONFIG["OUT_DIR"], "extract_meta.json"), "w") as f:
    json.dump({"prompt_version": P.PROMPT_VERSION, "prompt_md5": PROMPT_MD5,
               "model": P.MODEL_ID, "max_len": CONFIG["MAX_LEN"],
               "extract_point": "last prompt token (generation prompt end)",
               "n_rows": int(len(df)), "n_layers": int(N_LAYERS),
               "modes": CONFIG["MODES"],
               "label_source": "CSV kside (v1 closed-book 기준 — 재판정 전이면 혼합 조건)"},
              f, ensure_ascii=False, indent=2)
print("[done]", CONFIG["OUT_DIR"])

  [cq] 100/1224 (bs=4)
  [cq] 200/1224 (bs=4)
  [cq] 300/1224 (bs=4)
  [cq] 400/1224 (bs=4)
  [cq] 500/1224 (bs=4)
  [cq] 600/1224 (bs=4)
  [cq] 700/1224 (bs=4)
  [cq] 800/1224 (bs=4)
  [cq] 900/1224 (bs=4)
  [cq] 1000/1224 (bs=4)
  [cq] 1100/1224 (bs=4)
  [cq] 1200/1224 (bs=4)
[saved] /content/drive/MyDrive/p8s/E1a_v2_dy/hidden_cq.npz  H=(1224, 33, 4096)
[done] /content/drive/MyDrive/p8s/E1a_v2_dy


## 주의

이 노트북은 **hidden만 추출한다. K± 라벨 재판정은 하지 않는다.**
`kside`는 CSV 값(v1 closed-book 기준)을 그대로 복사하므로, 라벨 재판정 전에는
**v2 hidden + v1 라벨**의 혼합 조건이다. 결과 보고 시 이 점을 명시할 것.

In [17]:
!grep -io "hf_[A-Za-z0-9]\{20,\}" /content/E1a_extract_hidden_v2.ipynb

grep: /content/E1a_extract_hidden_v2.ipynb: No such file or directory
